In [1]:
%matplotlib inline

In [2]:
from essential.kegg_modules import list_kegg_modules, kegg_module_to_graph, metabolic_to_operational_graph
from essential.plot_pathways import plot_pathway_results, plot_metabolic_pathway
from essential.utils import PLOTNINE_DEFAULT_THEME_2
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotnine as gg
from tqdm import tqdm
import scanpy as sc
from essential.pathway_discontinuity import PathwayDiscontinuity


plt.rcParams["svg.fonttype"] = "none"

def plot_umap_genes(adata, genes):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(genes)]
    obs_subset["target"] = obs_subset["target"].astype(str)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point()
        + gg.geom_point(obs_subset, gg.aes(color="target"), size=2)
        + gg.theme_minimal()
    )
    return fig

def plot_umap_equiv_classes(adata, gene_to_class, class_color_mapping, plot_legend=True, point_size=1.5):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(gene_to_class.keys())]
    obs_subset["equivalence_class"] = obs_subset["target"].map(gene_to_class).sample(frac=1)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point(size=0.7, stroke=0)
        + gg.geom_point(obs_subset, gg.aes(color="equivalence_class"), size=point_size, stroke=0.0)
        + gg.scale_color_manual(values=class_color_mapping)
        + gg.theme_minimal()
    )
    if not plot_legend:
        fig = fig + gg.theme(legend_position="none")
    return fig

/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/themes/themeable.py:2538: FutureWarning: Themeable 'axis_ticks_direction' is deprecated andwill be removed in a future version. Use +ve or -ve values of the axis_ticks_lengthto affect the direction of the ticks.
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/themes/themeable.py:2566: FutureWarning: Themeable 'axis_ticks_direction' is deprecated andwill be removed in a future version. Use +ve/-ve/complex values of the axis_ticks_lengthto affect the direction of the ticks.
/Users/pboyeau/Projects/essential/src/essential/utils.py:21: FutureWarning: 
Themeable 'legend_entry_spacing' has been renamed to 'legend_key_spacing'.
'legend_entry_spacing' is now deprecated and will be removed in a future release.


#### Import transcriptomic data

In [3]:
adata = sc.read_h5ad("../../data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad")
# adata.X = adata.layers["reads"]
# adata = adata[~adata.obs["target"].isna()].copy()
# sc.pp.highly_variable_genes(adata, n_top_genes=500, flavor="seurat_v3")
# adata = adata[:, adata.var["highly_variable"]].copy()
# sc.pp.normalize_total(adata)
# sc.pp.log1p(adata)
# sc.pp.pca(adata, n_comps=50)
# For now, just take the PCA from James

sc.pp.neighbors(adata, n_neighbors=10, use_rep="X_pca")
sc.tl.umap(adata, min_dist=0.5)

adata.obs["UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["UMAP2"] = adata.obsm["X_umap"][:, 1]

/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [ ]:
# sc.tl.leiden(adata, resolution=0.1)
sc.tl.umap(adata)
sc.pl.umap(adata, color="leiden")

In [ ]:
from scvi.model import SCVI

model = SCVI(adata, gene_likelihood="nb")
model.fit()


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



ImportError: cannot import name 'SCVI' from 'scvi.model' (/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/scvi/model/__init__.py)

In [ ]:
adata.obs

In [ ]:
sc.tl.umap(adata, min_dist=0.5)
sc.pl.umap(adata)

#### Mine KEGG modules

In [ ]:
module_info = pd.DataFrame(list_kegg_modules("eco"))
for i, row in tqdm(module_info.iterrows()):
    module_name = row["module_id"]
    g = kegg_module_to_graph(module_name, 'eco')
    genes = np.unique([d['gene'] for u, v, d in g.edges(data=True) if d['gene']])
    np.sort(genes)
    genes_str = ', '.join(genes)

    op = metabolic_to_operational_graph(g)

    pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
    results = pda.fit(threshold=0.1, mode="mmd_stat")

    module_info.loc[i, "n_equivalences"] = results.n_equivalences
    module_info.loc[i, "module_has_surprises"] = results.n_equivalences >= 2
    module_info.loc[i, "genes"] = genes_str
    module_info.loc[i, "n_genes"] = len(genes)
module_info.to_csv("module_info.csv", index=False)
module_info.to_json("module_info.json", orient="records", indent=2)


module_prediction = pd.read_json("module_prediction.json")
module_info_ = module_info.merge(module_prediction, on="module_id", how="left").query("n_genes >= 2")

fig = (
    gg.ggplot(
        module_info_,
        gg.aes(x="factor(activity_prediction)", fill="factor(module_has_surprises)")
    ) 
    + gg.geom_bar(position="dodge", width=0.5) 
    + gg.labs(
        x="activity prediction",
        y="# of modules",
        fill="module has surprises"
    ) 
    + gg.theme_minimal()
    + gg.scale_y_continuous(expand=(0, 0))
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        figure_size=(2.7, 2)
    )
)
fig.save("figures/module_has_surprises_by_activity_prediction.svg")
fig

In [ ]:
DISPLAY_TOP_K_MODULES = 10

display(module_info_.query("activity_prediction == 'active'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

display(module_info_.query("activity_prediction == 'partially active'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

display(module_info_.query("activity_prediction == 'inactive'").sort_values("n_equivalences", ascending=False).head(DISPLAY_TOP_K_MODULES))

#### select and visualize a module

In [ ]:
EXAMPLES = [
    "M00938",  # dUTP toxicity,
    "M00120", # "coA biosynthesis"
    "M00063",  # CMP-KDO biosynthesis
    "M00121", # "heme
]

In [ ]:
module_info_.query("activity_prediction == 'active'").query("module_has_surprises").sort_values("n_equivalences", ascending=False)

In [ ]:
# module_name = "eco_M00003"
module_name = "eco_M00060"
g = kegg_module_to_graph(module_name, 'eco', print_info=True)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=0.1, mode="mmd_stat")
results

In [ ]:
fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False)
display(fig)

In [ ]:
# _ = plot_metabolic_pathway(g, figsize=(5, 5))

In [ ]:
%matplotlib inline

In [ ]:
pairs = results.gene_pair_scores.sort_values("score", ascending=False)
for _, row in pairs.iterrows():
    genes = [row["g1"], row["g2"]]

    fig = plot_umap_genes(adata, genes)
    fig = fig + gg.ggtitle(f"{row['g1']} - {row['g2']}: {row['score']:.2f}")
    display(fig)

# Individual modules

In [ ]:
THRESHOLD = 0.1

## A

In [ ]:
module_id = "eco_M00001"

g = kegg_module_to_graph(module_id, 'eco', print_info=False)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=THRESHOLD, mode="mmd_stat")

fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2.5, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)
plt.savefig(f"figures/eco_{module_id}.svg")

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False) + PLOTNINE_DEFAULT_THEME_2
fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)
display(fig)


## B

In [ ]:
module_id = "eco_M00121"

g = kegg_module_to_graph(module_id, 'eco', print_info=False)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=THRESHOLD, mode="mmd_stat")

fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2.5, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)
plt.savefig(f"figures/eco_{module_id}.svg")

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False) + PLOTNINE_DEFAULT_THEME_2
fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)
display(fig)


## C

In [ ]:
module_id = "eco_M00060"

g = kegg_module_to_graph(module_id, 'eco', print_info=False)
op = metabolic_to_operational_graph(g)

pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
results = pda.fit(threshold=THRESHOLD, mode="mmd_stat")

fig, ax, plot_info = plot_pathway_results(
    g, 
    results.edge_equivalence, 
    x_sep=0.5, 
    y_sep=2.5, 
    label_offset=(0, 0.4), 
    edge_width=1.7, 
    color_surprising_individually=True
)
display(fig)
plt.savefig(f"figures/eco_{module_id}.svg")

fig = plot_umap_equiv_classes(adata, plot_info["gene_to_class"], plot_info["class_color_mapping"], plot_legend=False) + PLOTNINE_DEFAULT_THEME_2
fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)
display(fig)


# Batched experiment

In [ ]:
selected_modules = module_info_.query("activity_prediction == 'active'").query("module_has_surprises").sort_values("n_equivalences", ascending=False)

In [14]:
for _, row in selected_modules.iterrows():
    module_id = row["module_id"]
    print(module_id)

    g = kegg_module_to_graph(module_id, 'eco', print_info=False)
    op = metabolic_to_operational_graph(g)

    pda = PathwayDiscontinuity(adata, representation_obsm_key="X_pca", metabolic_graph=op, perturbation_obs_key="target", global_sigma=5.0)
    results = pda.fit(threshold=0.2, mode="mmd_stat")

    fig, ax, plot_info = plot_pathway_results(
        g, 
        results.edge_equivalence, 
        x_sep=0.5, 
        y_sep=2, 
        label_offset=(0, 0.4), 
        edge_width=1.7, 
        color_surprising_individually=True
    )
    # display(fig)
    plt.savefig(f"figures/eco_{module_id}.svg")

    fig = plot_umap_equiv_classes(
        adata, 
        plot_info["gene_to_class"], 
        plot_info["class_color_mapping"], 
        plot_legend=False,
        point_size=5.0,
    )
    # display(fig)
    fig.save(f"figures/eco_{module_id}_umap.png", dpi=500)


eco_M00001


/Users/pboyeau/Projects/essential/src/essential/plot_pathways.py:466: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
/var/folders/lb/fgy8hrqj0kd6w1fzrws8bv1m0000gp/T/ipykernel_7773/2473423727.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning

eco_M00002


/var/folders/lb/fgy8hrqj0kd6w1fzrws8bv1m0000gp/T/ipykernel_7773/2473423727.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/eco_eco_M00002_umap.png


eco_M00125


...

/var/folders/lb/fgy8hrqj0kd6w1fzrws8bv1m0000gp/T/ipykernel_7773/2473423727.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/eco_eco_M00011_umap.png


eco_M00009


/var/folders/lb/fgy8hrqj0kd6w1fzrws8bv1m0000gp/T/ipykernel_7773/2473423727.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/eco_eco_M00009_umap.png


eco_M01025


/Users/pboyeau/Projects/essential/src/essential/pathway_discontinuity.py:178: UserWarning: Gene 'wecE' has no observations in adata.obs['target']. Returning NaN.
/var/folders/lb/fgy8hrqj0kd6w1fzrws8bv1m0000gp/T/ipykernel_7773/2473423727.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/pboyeau/miniforge3/envs/essential/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/eco_eco_M01025_umap.png


In [ ]:
eco_M00001